In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

# Notebook 06 · Trazabilidad de baseline (Day 02)

- En este notebook se reproducen las decisiones del proyecto referentes a la comparación entre modelos a partir del baseline congelado (lock) en reportes/artefactos oficiales del proyecto.
- Se documentan las decisiones del Day 02 sobre gobernanza de métricas, comparación oficial de modelos.
- El baseline oficial del proyecto es `LR_smote_0.5` (artefacto bloqueado y trazable en reportes de lock).

## `final_baseline_vs_candidates.csv`

- Es el acta oficial de comparación entre baseline y candidatos del proyecto.
- Se guarda en `artifacts/public/metrics/final_baseline_vs_candidates.csv`.
- Grano: 1 fila por evaluación oficial (`run_id` + `model_variant` + `cutoff_date`).
- Incluye baseline (`LR_smote_0.5`) como fila de referencia.
- Cada candidato nuevo se añade como nueva fila para comparar deltas y decisión de promoción.

## Decisión D02-001 · Acta única de comparación

- El CSV `artifacts/public/metrics/final_baseline_vs_candidates.csv` servirá como artefacto único para comparar candidatos de manera desambiguada y auditable. El criterio es reproduccibilidad, trazabilidad y comparación consistente estandarizada contra baseline.
- Cualquier promoción/rechazo de modelo debe estar respaldada por una fila en este CSV.

### CSV

**Columnas mínimas cerradas**
| columna | para qué sirve |
|---|---|
| `run_id` | identificador único de corrida |
| `day_id` | día del plan (Day 02, Day 03...) |
| `model_variant` | nombre del modelo (`LR_smote_0.5`, `V3_A`, etc.) |
| `model_role` | `baseline` o `candidate` |
| `dataset_name` | dataset usado |
| `dataset_snapshot_hash` | trazabilidad de datos |
| `cutoff_date` | corte temporal del split |
| `test_events` | soporte real de evaluación |
| `accuracy` | métrica fila |
| `balanced_accuracy` | métrica fila clave |
| `f1_pos` | métrica fila clase positiva |
| `top1_hit` | métrica evento |
| `top2_hit` | métrica evento principal |
| `coverage` | cobertura operativa |
| `delta_top2_vs_baseline` | mejora/empeora vs baseline |
| `delta_bal_acc_vs_baseline` | mejora/empeora vs baseline |
| `gate_pass` | pasa/no pasa el gate |
| `promotion_decision` | `promote` o `keep_baseline` |
| `model_path` | ruta del artefacto |
| `metadata_path` | ruta de metadata |

**Ejemplo**
```csv
run_id,day_id,model_variant,model_role,dataset_name,cutoff_date,accuracy,balanced_accuracy,f1_pos,top1_hit,top2_hit,coverage,delta_top2_vs_baseline,delta_bal_acc_vs_baseline,gate_pass,promotion_decision
20260304T120000Z,Day 02,LR_smote_0.5,baseline,dataset_modelo_proveedor_v2_candidates.csv,2028-02-21,0.9095,0.8615,0.6899,0.7721,0.8629,0.9820,0.0000,0.0000,NA,keep_baseline
20260305T101500Z,Day 03,V3_A,candidate,dataset_modelo_proveedor_v3a.csv,2028-02-21,0.9110,0.8670,0.7000,0.7750,0.8700,0.9800,0.0071,0.0055,false,keep_baseline
```

### SCRIPT

- Actualización automatizada implementada mediante `src/ml/final_baseline_registry.py` (`init-baseline` y `append-candidate`).

## Decisión D02-002 · Contrato exacto de `metrics-json` para `append-candidate`

- `metrics-json` es el archivo de métricas que consume el comando `append-candidate` para añadir un candidato al acta oficial.
- Este contrato define qué campos mínimos debe traer ese JSON para que la comparación contra baseline sea consistente.
- Formatos aceptados:
  - Métricas en raíz del JSON.
  - Métricas dentro de `metrics`.

Campos mínimos:
- `accuracy` (o `test_acc`)
- `balanced_accuracy` (o `test_bal_acc` / `bal_acc`)
- `f1_pos` (o `test_f1_pos`)
- `top1_hit`
- `top2_hit`
- `coverage`
- `test_events`

### Regla

- Si falta alguno de los campos clave para gate (`top2_hit`, `balanced_accuracy`, `coverage`), la promoción no se decide en automático.


```json
{
  "accuracy": 0.9123,
  "balanced_accuracy": 0.8734,
  "f1_pos": 0.7021,
  "top1_hit": 0.7810,
  "top2_hit": 0.8800,
  "coverage": 0.9950,
  "test_events": 2633
}
```

## Decisión D02-003 · Definición oficial de `coverage`

- `coverage` = porcentaje de eventos válidos para los que el sistema entrega recomendación usable.
- Fórmula: `eventos_con_recomendacion / eventos_validos`.
- Escala: valor entre `0` y `1`.

Regla de gate en este proyecto:
- El candidato no debe caer más de `0.005` vs baseline en cobertura (equivale a 0.5 puntos porcentuales).
- Si no hay `coverage` en baseline o candidato, no se aprueba promoción automática.

Nota operativa:
- Para decisiones automáticas de gate, `coverage` debe venir informado tanto en baseline como en candidato.

# Reproducibilidad oficial Day 02

```bash
# 1) Inicializar acta oficial con baseline (una sola vez)
python3 src/ml/final_baseline_registry.py init-baseline \
  --output artifacts/public/metrics/final_baseline_vs_candidates.csv \
  --run-id 20260304T120000Z \
  --day-id "Day 02" \
  --model-variant LR_smote_0.5 \
  --metadata models/public/baseline/metadata.json \
  --metrics-json artifacts/public/metrics/baseline/20260304_baseline_metrics.json \
  --dataset data/public/dataset_modelo_proveedor_v2_candidates.csv \
  --coverage 1.0 \
  --test-events 2633 \
  --overwrite

# 2) Añadir candidato (cada nueva evaluación Day 03+)
python3 src/ml/final_baseline_registry.py append-candidate \
  --output artifacts/public/metrics/final_baseline_vs_candidates.csv \
  --run-id <RUN_ID> \
  --day-id "Day 03" \
  --model-variant <MODEL_VARIANT> \
  --metadata <RUTA_METADATA_CANDIDATO> \
  --metrics-json <RUTA_METRICS_JSON_CANDIDATO> \
  --dataset <RUTA_DATASET_EVALUADO>
```